## Chunked embeddings → Qdrant (resumes & job postings)

This notebook walks `data/cv-dataset-processed` and `data/job-postings-processed`, turns each JSON into text, splits it into **≤512-token** overlapping chunks, obtains **1024-dim** embeddings through the BAML function `EmbedChunkLlamaCppPC` (`baml_src/embeddings.baml`, client `LlamaCppPCembedding` in `baml_src/clients.baml`), **pools** chunk vectors using rules analogous to [`STRATEGY.md`](../STRATEGY.md), and writes **one Qdrant collection per document**:

- Collections: `resume-<domain>-<id>` or `job-posting-<domain>-<id>` where `<domain>` matches the subdirectory name and `<id>` is the JSON stem (same as preprocessing). Each collection holds **six Qdrant points** (same 1024-dim cosine vector space), one per pooling strategy mapped from `STRATEGY.md`: `pooling_method` is `max_score`, `mean_all`, `topk_mean_k5`, `weighted_topk_mean_k5`, `softmax_pooling_alpha10`, or `hybrid_max_topk_k5_w0_4`. At query time, filter by payload or restrict search to IDs `0`–`5` depending on convention above.

**Requirements**

- `./run-llama-servers.sh` (or equivalent): embedding HTTP API on **`127.0.0.1:8081`** (`mxbai-embed-large`).
- Qdrant at `http://localhost:6333` (see [`ollama_connection.ipynb`](ollama_connection.ipynb)).
- **`uv sync`** after pulling deps (includes `tiktoken` for counting tokens; fallback uses a coarse estimate if unavailable).

Then run **`baml-cli generate --from ./baml_src`** from the repo root whenever BAML sources change (`AGENTS.md`).

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from baml_client.sync_client import b

# Tiktoken sliding windows (~512-token chunks); falls back to char-based slicing if unavailable.
_HAS_TIKTOKEN = False

try:
    import tiktoken

    _tik_enc = tiktoken.get_encoding("cl100k_base")
    _HAS_TIKTOKEN = True
except ImportError:
    _tik_enc = None


CV_DIR = ROOT / "data" / "cv-dataset-processed"
JOB_DIR = ROOT / "data" / "job-postings-processed"

QDRANT_URL = "http://localhost:6333"
EMBED_DIM = 1024

MAX_CHUNK_TOKENS = 512
TOKEN_OVERLAP = 64

TOP_K = 5
SOFTMAX_ALPHA = 10.0
HYBRID_W = 0.4

In [ ]:
def chunk_text_by_tokens(
    text: str,
    max_tokens: int = MAX_CHUNK_TOKENS,
    overlap_tokens: int = TOKEN_OVERLAP,
) -> list[str]:
    """Windows of up to ``max_tokens``, advancing by ``max_tokens - overlap_tokens``."""

    text = text.strip()
    if not text:
        return []

    overlap_tokens = max(0, min(overlap_tokens, max_tokens - 1))
    step = max(1, max_tokens - overlap_tokens)

    if _HAS_TIKTOKEN and _tik_enc is not None:
        ids = _tik_enc.encode(text)
        chunks: list[str] = []

        for i in range(0, len(ids), step):
            span = ids[i : i + max_tokens]
            decoded = _tik_enc.decode(span).strip()
            if decoded:
                chunks.append(decoded)

        return chunks if chunks else [_tik_enc.decode(ids[:max_tokens]).strip()]

    # Approximate fallback when tiktoken is not installed (~4 chars / token).
    approx_cpt = 4
    max_chars = max(1, max_tokens * approx_cpt)
    overlap_chars = overlap_tokens * approx_cpt
    stride = max(1, max_chars - overlap_chars)
    chunks = []
    n = len(text)
    start = 0

    while start < n:
        end = min(n, start + max_chars)
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= n:
            break
        start += stride

    return chunks if chunks else [text]

In [ ]:
def _l2_normalize(v: np.ndarray) -> np.ndarray:
    n = float(np.linalg.norm(v))
    if n == 0:
        return v
    return v / n


def _numpy_chunk_vectors(embeddings: list[list[float]]) -> np.ndarray:
    return np.asarray(embeddings, dtype=np.float64)


def chunk_salience(emb: np.ndarray) -> float:
    """Proxy for STRATEGY chunk scores when no query exists (L2 norm — model-specific)."""

    return float(np.linalg.norm(emb))


def pool_mean_all(vectors: np.ndarray) -> np.ndarray:
    return _l2_normalize(vectors.mean(axis=0))


def pool_max_score(vectors: np.ndarray, salience: np.ndarray) -> np.ndarray:
    idx = int(np.argmax(salience))
    return _l2_normalize(vectors[idx])


def pool_topk_mean(vectors: np.ndarray, salience: np.ndarray, k: int = TOP_K) -> np.ndarray:
    k_eff = max(1, min(k, len(vectors)))
    order = np.argsort(-salience)[:k_eff]
    return _l2_normalize(vectors[order].mean(axis=0))


def pool_weighted_topk_mean(vectors: np.ndarray, salience: np.ndarray, k: int = TOP_K) -> np.ndarray:
    k_eff = max(1, min(k, len(vectors)))
    order = np.argsort(-salience)[:k_eff]
    weights = np.arange(k_eff, 0, -1, dtype=np.float64)
    weighted = np.average(vectors[order], axis=0, weights=weights)
    return _l2_normalize(weighted)


def pool_softmax(vectors: np.ndarray, salience: np.ndarray, alpha: float = SOFTMAX_ALPHA) -> np.ndarray:
    s = salience.astype(np.float64)
    stabilized = alpha * (s - s.max())
    w = np.exp(stabilized)
    w /= w.sum()
    return _l2_normalize((w[:, None] * vectors).sum(axis=0))


def pool_hybrid(vectors: np.ndarray, salience: np.ndarray, k: int = TOP_K, w: float = HYBRID_W) -> np.ndarray:
    best_idx = int(np.argmax(salience))
    v_max = vectors[best_idx]
    v_topk = pool_topk_mean(vectors, salience, k=k)
    blended = w * v_max + (1.0 - w) * v_topk
    return _l2_normalize(blended)


def build_pooled_vectors(chunk_embeddings: list[list[float]]) -> dict[str, np.ndarray]:
    mats = _numpy_chunk_vectors(chunk_embeddings)

    sal = np.array([chunk_salience(mats[i]) for i in range(len(mats))], dtype=np.float64)

    return {
        "max_score": pool_max_score(mats, sal),
        "mean_all": pool_mean_all(mats),
        "topk_mean_k5": pool_topk_mean(mats, sal, k=TOP_K),
        "weighted_topk_mean_k5": pool_weighted_topk_mean(mats, sal, k=TOP_K),
        "softmax_pooling_alpha10": pool_softmax(mats, sal, alpha=SOFTMAX_ALPHA),
        "hybrid_max_topk_k5_w0_4": pool_hybrid(mats, sal, k=TOP_K, w=HYBRID_W),
    }

In [ ]:
def list_json_docs(base: Path) -> list[tuple[str, Path]]:
    rows: list[tuple[str, Path]] = []
    if not base.is_dir():
        return rows

    for domain_dir in sorted(p for p in base.iterdir() if p.is_dir()):
        domain = domain_dir.name
        for jf in sorted(domain_dir.glob("*.json")):
            rows.append((domain, jf))
    return rows


def sanitize_collection_fragment(s: str) -> str:
    # Qdrant: letters, digits, hyphens commonly used; folders already match preprocessing.
    return s.replace("/", "-").strip()


def collection_name(kind: str, domain: str, stem: str) -> str:
    """kind: resume | job-posting."""

    frag_domain = sanitize_collection_fragment(domain)
    frag_id = sanitize_collection_fragment(stem)
    return f"{kind}-{frag_domain}-{frag_id}"


def embed_chunks(chunks: list[str]) -> list[list[float]]:
    out: list[list[float]] = []
    for chunk in chunks:
        vec = b.EmbedChunkLlamaCppPC(chunk)

        if len(vec) != EMBED_DIM:
            raise RuntimeError(f"unexpected embedding dimension {len(vec)} (expected {EMBED_DIM})")
        out.append(vec)
    return out


def ensure_collection(client: QdrantClient, name: str, overwrite: bool) -> None:
    names = {c.name for c in client.get_collections().collections}
    exists = name in names

    if exists and overwrite:
        client.delete_collection(name)
        exists = False

    if not exists:
        client.create_collection(
            collection_name=name,
            vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
        )


METHOD_ORDER = (
    "max_score",
    "mean_all",
    "topk_mean_k5",
    "weighted_topk_mean_k5",
    "softmax_pooling_alpha10",
    "hybrid_max_topk_k5_w0_4",
)


def upsert_document_poolings(
    client: QdrantClient,
    collection: str,
    pooled: dict[str, np.ndarray],
    payload_base: dict,
) -> None:
    """One point per pooling method (distinct named vectors alternative is heavier ops)."""

    points: list[PointStruct] = []

    for i, key in enumerate(METHOD_ORDER):
        if key not in pooled:
            continue

        vec = pooled[key].astype(float).tolist()
        pid = i

        payload = {
            **payload_base,
            "pooling_method": key,
            "embedding_dim": EMBED_DIM,
        }

        points.append(PointStruct(id=pid, vector=vec, payload=payload))

    if not points:
        return

    client.upsert(collection_name=collection, wait=True, points=points)


def process_documents(
    client: QdrantClient,
    items: list[tuple[str, str, Path]],
    kind_prefix: str,
    overwrite: bool = True,
):
    """

    Parameters
    ----------
    items :
        tuples of `(domain, json_stem_without_suffix, json_path)`.
    kind_prefix :
        `"resume"` or `"job-posting"` for naming.
    """

    for domain, stem, path in items:
        col = collection_name(kind_prefix, domain, stem)

        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        text = json.dumps(data, ensure_ascii=False, sort_keys=True, indent=2)

        chunks = chunk_text_by_tokens(text)

        if not chunks:
            print(f"[skip empty] {col}")
            continue

        print(f"[embed] {col} ({len(chunks)} chunks)")
        embeddings = embed_chunks(chunks)

        pooled = build_pooled_vectors(embeddings)

        ensure_collection(client, col, overwrite=overwrite)

        upsert_document_poolings(
            client,
            col,
            pooled,
            payload_base={
                "kind": kind_prefix,
                "domain": domain,
                "document_id": stem,
                "source_path": str(path.relative_to(ROOT)),
                "chunk_count": len(chunks),
            },
        )

In [ ]:
OVERWRITE_COLLECTIONS = True

qdrant = QdrantClient(url=QDRANT_URL)

cv_items = []

for domain, jp in list_json_docs(CV_DIR):
    stem = jp.stem
    cv_items.append((domain, stem, jp))


job_items = []

for domain, jp in list_json_docs(JOB_DIR):
    stem = jp.stem
    job_items.append((domain, stem, jp))


print(f"resume JSON files: {len(cv_items)}")
print(f"job posting JSON files: {len(job_items)}")

In [ ]:
process_documents(qdrant, cv_items, "resume", overwrite=OVERWRITE_COLLECTIONS)

In [ ]:
process_documents(qdrant, job_items, "job-posting", overwrite=OVERWRITE_COLLECTIONS)